# Titanic: очистка данных

Данные взяты из Kaggle (Titanic, файл `train.csv`, в проекте он называется `train-2.csv`). Ноутбук готовит данные к анализу: закрывает пропуски, добавляет новые признаки, убирает ненужные столбцы и сохраняет результат в `train_clean.csv`.

Пропуски в исходных данных: `Age` — 177, `Cabin` — 687 (из 891), `Embarked` — 2.

## Шаг 1. Загрузка и поиск пропусков

Читаю файл и работаю с копией, чтобы исходные данные остались нетронутыми. Считаю пропуски по столбцам, чтобы понять, что именно нужно чинить.

In [1]:
import pandas as pd

raw = pd.read_csv('train-2.csv')
df = raw.copy()
print(df.shape)
print(df.isna().sum())

(891, 12)
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Шаг 2. Первичный осмотр

Проверяю дубликаты, распределение по полу, классу и порту посадки, а также диапазон возраста и цены билета. Это нужно, чтобы убедиться, что в данных нет явных аномалий.

In [2]:
print(df.duplicated().sum())
print(df['Sex'].value_counts())
print(df['Pclass'].value_counts())
print(df['Embarked'].value_counts(dropna=False))
print(df['Age'].agg(['min', 'max']))
print(df['Fare'].agg(['min', 'max']))

0
Sex
male      577
female    314
Name: count, dtype: int64
Pclass
3    491
1    216
2    184
Name: count, dtype: int64
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64
min     0.42
max    80.00
Name: Age, dtype: float64
min      0.0000
max    512.3292
Name: Fare, dtype: float64


## Шаг 3. Пропуски в Embarked

Пропусков всего два, поэтому заполняю их самым частым портом посадки — на общую картину это почти не повлияет.

In [3]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print(df['Embarked'].isna().sum())
print(df['Embarked'].value_counts())

0
Embarked
S    646
C    168
Q     77
Name: count, dtype: int64


## Шаг 4. Пропуски в Age

Пропусков 177 — слишком много, чтобы удалять строки. Общая медиана возраста грубая, поэтому смотрю медиану по классу и полу: она сильно различается между группами.

In [4]:
print(df['Age'].median())
print(df.groupby(['Pclass', 'Sex'])['Age'].median())

28.0
Pclass  Sex   
1       female    35.0
        male      40.0
2       female    28.0
        male      30.0
3       female    21.5
        male      25.0
Name: Age, dtype: float64


Заполняю пропуски в `Age` медианой, рассчитанной по группам «класс + пол». Проверяю, что пропусков не осталось.

In [5]:
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Sex'])['Age'].transform('median'))

print(df['Age'].isna().sum())

0


## Шаг 5. Cabin: признак HasCabin

В `Cabin` 687 пропусков из 891 — заполнять нечем. Сохраняю саму информацию «каюта известна или нет» в новый признак `HasCabin`, а столбец `Cabin` удаляю.

In [6]:
df['HasCabin'] = df['Cabin'].notna()
df = df.drop(columns=['Cabin'])
print(df['HasCabin'].sum())

204


## Шаг 6. Состав семьи

Считаю размер семьи (`SibSp` + `Parch` + сам пассажир) и отмечаю тех, кто плыл один (`IsAlone`). Эти признаки обычно полезнее исходных `SibSp` и `Parch` по отдельности.

In [7]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = df['FamilySize'] == 1
print(df['FamilySize'].max())
print(df['IsAlone'].sum())

11
537


## Шаг 7. Удаление лишнего и сохранение

Убеждаюсь, что пропусков не осталось. Удаляю `Ticket` и `PassengerId`: это идентификаторы, для анализа они не нужны. Сохраняю результат в `train_clean.csv` и перечитываю файл для проверки.

In [8]:
print(df.isna().sum().sum())
df = df.drop(columns=['Ticket', 'PassengerId'])
df.to_csv('train_clean.csv', index=False)

check = pd.read_csv('train_clean.csv')
print(check.shape)
print(list(check.columns))

0
(891, 12)
['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'HasCabin', 'FamilySize', 'IsAlone']


## Проверка итога

Сравниваю размеры данных до и после очистки.

In [9]:
print('До:   ', raw.shape)
print('После:', check.shape)
print('Пропусков осталось:', check.isna().sum().sum())

До:    (891, 12)
После: (891, 12)
Пропусков осталось: 0


## Итог

- **Размер:** было 891 строка и 12 столбцов, стало 891 строка и 12 столбцов (3 удалено, 3 добавлено). Строки не удалялись.
- **Пропуски закрыты:**
  - `Embarked` (2) — самым частым портом;
  - `Age` (177) — медианой по классу и полу;
  - `Cabin` (687) — столбец заменён признаком `HasCabin`.
- **Добавлены признаки:** `HasCabin`, `FamilySize`, `IsAlone`.
- **Удалены столбцы:** `Cabin`, `Ticket`, `PassengerId`.
- **Результат:** `train_clean.csv` без пропусков.